In [15]:
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
import timm

# Image size for ViT
image_size = 224

# Device setup
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

In [16]:
# Enhanced augmentations for training
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(image_size, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomRotation(20),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), shear=10),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Simpler transform for validation/test
test_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [17]:
class BirdDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.data = pd.read_csv(csv_file)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = self.data.iloc[idx]['image_path']
        label = int(self.data.iloc[idx]['label']) - 1  # Shift 1–200 to 0–199 here
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [18]:
# Load and split the dataset
df = pd.read_csv('train_labels.csv')
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)

train_df.to_csv('train_split.csv', index=False)
val_df.to_csv('val_split.csv', index=False)

# Create datasets
train_dataset = BirdDataset(csv_file='train_split.csv', transform=train_transform)
val_dataset = BirdDataset(csv_file='val_split.csv', transform=test_transform)

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4)

In [19]:
# Load pretrained ViT and modify head
model = timm.create_model('vit_large_patch16_224', pretrained=True)

# Add dropout and new head for 200 classes
model.head = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.head.in_features, 200)
)

# Move to device
model = model.to(device)

# Freeze backbone initially for fine-tuning
for param in model.parameters():
    param.requires_grad = False
for param in model.head.parameters():
    param.requires_grad = True

In [20]:
# Loss function with label smoothing
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Optimizer with layer-wise learning rates
optimizer = optim.AdamW([
    {'params': model.head.parameters(), 'lr': 1e-3},
    {'params': model.blocks.parameters(), 'lr': 1e-4, 'requires_grad': False},  # Unfrozen later
    {'params': model.patch_embed.parameters(), 'lr': 1e-5, 'requires_grad': False}
])

# Scheduler
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

In [21]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, device, num_epochs=20, patience=3, unfreeze_epoch=5):
    best_acc = 0.0
    patience_counter = 0

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        
        # Unfreeze backbone layers (Transformer blocks) after a few epochs
        if epoch == unfreeze_epoch:
            print("🔓 Unfreezing backbone layers")
            for param in model.blocks.parameters():
                param.requires_grad = True
            for param in model.patch_embed.parameters():
                param.requires_grad = True
        
        # Training phase
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss += loss.item() * images.size(0)

        scheduler.step()

        # Validation phase
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_acc = correct / total
        train_loss = running_loss / len(train_loader.dataset)
        print(f"Train Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}")

        # Early stopping
        if val_acc > best_acc:
            best_acc = val_acc
            patience_counter = 0
            torch.save(model.state_dict(), 'best_model.pth')
            print("✅ Saved new best model")
        else:
            patience_counter += 1
            print(f"⚠️ Early stopping patience: {patience_counter}/{patience}")
            if patience_counter >= patience:
                print("⏹️ Early stopping triggered.")
                break

    return best_acc

In [22]:
def evaluate_with_tta(model, val_loader, device):
    model.eval()
    correct = 0
    total = 0
    
    # Define TTA transforms
    tta_transforms = [
        test_transform,
        transforms.Compose([transforms.RandomHorizontalFlip(p=1), *test_transform.transforms[1:]]),
        transforms.Compose([transforms.RandomRotation(10), *test_transform.transforms[1:]])
    ]
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            batch_preds = []
            
            # Apply TTA
            for t in tta_transforms:
                aug_images = torch.stack([t(Image.fromarray(img.permute(1, 2, 0).cpu().numpy())) 
                                        for img in images])
                aug_images = aug_images.to(device)
                outputs = model(aug_images)
                batch_preds.append(outputs)
            
            # Average predictions
            avg_outputs = torch.mean(torch.stack(batch_preds), dim=0)
            _, predicted = torch.max(avg_outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    tta_acc = correct / total
    print(f"TTA Validation Accuracy: {tta_acc:.4f}")
    return tta_acc

In [23]:
def evaluate_with_tta(model, val_loader, device):
    model.eval()
    correct = 0
    total = 0
    
    # Define TTA transforms
    tta_transforms = [
        test_transform,
        transforms.Compose([transforms.RandomHorizontalFlip(p=1), *test_transform.transforms[1:]]),
        transforms.Compose([transforms.RandomRotation(10), *test_transform.transforms[1:]])
    ]
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            batch_preds = []
            
            # Apply TTA
            for t in tta_transforms:
                aug_images = torch.stack([t(Image.fromarray(img.permute(1, 2, 0).cpu().numpy())) 
                                        for img in images])
                aug_images = aug_images.to(device)
                outputs = model(aug_images)
                batch_preds.append(outputs)
            
            # Average predictions
            avg_outputs = torch.mean(torch.stack(batch_preds), dim=0)
            _, predicted = torch.max(avg_outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    tta_acc = correct / total
    print(f"TTA Validation Accuracy: {tta_acc:.4f}")
    return tta_acc

In [ ]:
# Train the model
best_acc = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, device, num_epochs=20, patience=3)

# Load best model
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

# Evaluate with TTA
tta_acc = evaluate_with_tta(model, val_loader, device)
print(f"Best Standard Accuracy: {best_acc:.4f}")
print(f"Best TTA Accuracy: {tta_acc:.4f}")


Epoch 1/20
